# 05 - Deploy the classifier endpoint

Build plane, `deploy_endpoint` task of the `msd_training` job.

* Resolves the `champion` alias to an **exact** model version (the endpoint is always pinned to
  a version, never to "latest").
* Creates the Model Serving endpoint if it does not exist, or updates it when the champion
  changed. Refuses to touch an endpoint that serves a different model.
* Compute: `CPU` workload, `Small` size (4 concurrent requests), scale-to-zero enabled (first
  request after idle has a cold start). The model has no GPU requirement.
* Waits until the endpoint is `READY`, then smoke-tests it with one test image per class.

In [0]:
dbutils.widgets.text("catalog", "poc_metal_surface_defect_diagnostics")
dbutils.widgets.text("schema", "metal_surface_defects")
dbutils.widgets.text("model_name", "msd_defect_classifier")
dbutils.widgets.text("serving_endpoint", "msd-defect-classifier")
dbutils.widgets.text("ready_timeout_minutes", "45")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
registered_model_name = f"{catalog}.{schema}.{dbutils.widgets.get('model_name')}"
endpoint_name = dbutils.widgets.get("serving_endpoint")
ready_timeout_s = int(dbutils.widgets.get("ready_timeout_minutes")) * 60

CHAMPION = "champion"
WORKLOAD_TYPE, WORKLOAD_SIZE, SCALE_TO_ZERO = "CPU", "Small", True
print(f"endpoint: {endpoint_name}\nmodel: {registered_model_name}@{CHAMPION}")

In [0]:
from mlflow import MlflowClient

registry = MlflowClient(registry_uri="databricks-uc")
try:
    champion = registry.get_model_version_by_alias(registered_model_name, CHAMPION)
except Exception as e:
    raise RuntimeError(
        f"No @{CHAMPION} version for {registered_model_name}. The evaluate gate has not promoted any "
        "version yet, so there is nothing to deploy."
    ) from e
model_version = str(champion.version)
print(f"@{CHAMPION} -> version {model_version}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    EndpointTag,
    ServedEntityInput,
    ServingModelWorkloadType,
)

w = WorkspaceClient()

supported = {
    c["workload_type"]: [s["key"] for s in c.get("workload_sizes", [])]
    for c in w.api_client.do("GET", "/api/2.0/serving-endpoints:workload-configs").get("workload_configs", [])
}
if WORKLOAD_SIZE not in supported.get(WORKLOAD_TYPE, []):
    raise RuntimeError(f"{WORKLOAD_TYPE}/{WORKLOAD_SIZE} not supported in this workspace: {supported}")

served_entity = ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=model_version,
    workload_type=ServingModelWorkloadType(WORKLOAD_TYPE),
    workload_size=WORKLOAD_SIZE,
    scale_to_zero_enabled=SCALE_TO_ZERO,
)

try:
    existing = w.serving_endpoints.get(endpoint_name)
except NotFound:
    existing = None

if existing is None:
    w.serving_endpoints.create(
        name=endpoint_name,
        config=EndpointCoreConfigInput(name=endpoint_name, served_entities=[served_entity]),
        tags=[EndpointTag(key="project", value="metal_surface_defect_diagnostics")],
    )
    print(f"creating {endpoint_name} with v{model_version} (usually ~15 minutes)")
else:
    entities = list(existing.config.served_entities or []) if existing.config else []
    names = [e.entity_name for e in entities]
    if len(entities) != 1 or names[0] != registered_model_name:
        raise RuntimeError(f"Refusing to replace {endpoint_name}: it serves {names}, not {registered_model_name}.")
    if str(entities[0].entity_version) == model_version:
        print(f"{endpoint_name} already serves v{model_version}; no update needed")
    else:
        w.serving_endpoints.update_config(name=endpoint_name, served_entities=[served_entity])
        print(f"updating {endpoint_name}: v{entities[0].entity_version} -> v{model_version}")

In [0]:
import time

from databricks.sdk.service.serving import EndpointStateConfigUpdate, EndpointStateReady


def wait_for_endpoint_ready(name, timeout_s, poll_s=30):
    deadline = time.time() + timeout_s
    failures = {EndpointStateConfigUpdate.UPDATE_FAILED, EndpointStateConfigUpdate.UPDATE_CANCELED}
    while time.time() < deadline:
        state = w.serving_endpoints.get(name).state
        if state.ready == EndpointStateReady.READY and state.config_update == EndpointStateConfigUpdate.NOT_UPDATING:
            return
        if state.config_update in failures:
            raise RuntimeError(f"{name} deployment failed: {state.config_update.value}. Check the served model build/event logs.")
        time.sleep(poll_s)
    raise TimeoutError(f"{name} not ready after {timeout_s // 60} minutes")


wait_for_endpoint_ready(endpoint_name, ready_timeout_s)
served = w.serving_endpoints.get(endpoint_name).config.served_entities[0]
assert str(served.entity_version) == model_version, f"endpoint serves v{served.entity_version}, expected v{model_version}"
print(f"{endpoint_name} READY, serving {registered_model_name} v{model_version}")

In [0]:
import base64

sample = spark.sql(f"""
    SELECT label, content FROM (
      SELECT i.label, b.content, row_number() OVER (PARTITION BY i.label ORDER BY i.file_name) AS rn
      FROM {catalog}.{schema}.image_index i
      JOIN {catalog}.{schema}.image_bronze b ON i.path = regexp_replace(b.path, '^dbfs:', '')
      WHERE i.split = 'test'
    ) WHERE rn = 1 ORDER BY label
""").toPandas()

response = w.serving_endpoints.query(
    name=endpoint_name,
    dataframe_records=[{"image": base64.b64encode(b).decode()} for b in sample["content"]],
)
predictions = response.predictions
assert len(predictions) == len(sample), f"expected {len(sample)} predictions, got {len(predictions)}"
assert all({"predicted_class", "confidence", "probabilities"} <= set(p) for p in predictions), predictions[0]

correct = 0
for label, p in zip(sample["label"], predictions):
    correct += p["predicted_class"] == label
    print(f"{label:>16} -> {p['predicted_class']:<16} {p['confidence']:.3f}")
assert correct >= len(sample) - 1, f"only {correct}/{len(sample)} smoke-test images classified correctly"
print(f"smoke test OK: {correct}/{len(sample)} correct")